In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

df = pd.read_csv("C:\\Users\\saqli\\Downloads\\workspace\\Employee-Attrition-Project\\data\\employee_attrition_clean.csv")

print("Dataset shape:", df.shape)
print(df.head())
print(df["Attrition"].value_counts())

Dataset shape: (1000, 8)
   EmployeeID  TotalMonthsOfExp  TotalOrgsWorked  MonthsInOrg  \
0           1               110                4            9   
1           2               103                3           51   
2           3                41                4           16   
3           4                32                4           17   
4           5                80                3           16   

   LastPayIncrementBand  AverageFeedback  LastPromotionYears  Attrition  
0                     5                4                   4          1  
1                     1                4                   2          0  
2                     5                4                   4          1  
3                     5                2                   3          0  
4                     3                4                   2          0  
Attrition
0    770
1    230
Name: count, dtype: int64


In [2]:
X = df.drop(
    columns=["EmployeeID", "Attrition", "LastPromotionYears"]
)

y = df["Attrition"]

print("Features:", X.columns.tolist())
print("Target distribution:\n", y.value_counts())

Features: ['TotalMonthsOfExp', 'TotalOrgsWorked', 'MonthsInOrg', 'LastPayIncrementBand', 'AverageFeedback']
Target distribution:
 Attrition
0    770
1    230
Name: count, dtype: int64


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

Training rows: 800
Testing rows: 200


In [4]:
model = Pipeline([
    ("scaler", StandardScaler()),
    ("logistic_regression", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

model.fit(X_train, y_train)

print("Logistic Regression training completed.")

Logistic Regression training completed.


In [5]:
y_pred = model.predict(X_test)

# Probability of class 1
y_prob = model.predict_proba(X_test)[:, 1]

results = pd.DataFrame({
    "Actual": y_test.to_numpy(),
    "Predicted": y_pred,
    "Probability_Class_1": y_prob
})

print(results.head(10).round(3))

   Actual  Predicted  Probability_Class_1
0       0          0                0.355
1       0          1                0.599
2       0          0                0.369
3       0          0                0.366
4       0          0                0.428
5       0          0                0.361
6       1          0                0.455
7       0          0                0.464
8       0          0                0.451
9       0          0                0.356


In [6]:
probabilities = model.predict_proba(X_test)

print("First 5 probability pairs:")
print(probabilities[:5])

First 5 probability pairs:
[[0.64479079 0.35520921]
 [0.40082837 0.59917163]
 [0.63116108 0.36883892]
 [0.63373566 0.36626434]
 [0.5719796  0.4280204 ]]


In [7]:
threshold = 0.5

y_pred_threshold = (
    y_prob >= threshold
).astype(int)

print("Threshold:", threshold)
print(confusion_matrix(y_test, y_pred_threshold))
print(classification_report(
    y_test, y_pred_threshold, zero_division=0
))

Threshold: 0.5
[[78 76]
 [20 26]]
              precision    recall  f1-score   support

           0       0.80      0.51      0.62       154
           1       0.25      0.57      0.35        46

    accuracy                           0.52       200
   macro avg       0.53      0.54      0.49       200
weighted avg       0.67      0.52      0.56       200



In [8]:
print(
    "Accuracy:",
    round(accuracy_score(y_test, y_pred_threshold) * 100, 2),
    "%"
)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_threshold))

print("\nClassification Report:")
print(classification_report(
    y_test, y_pred_threshold, zero_division=0
))

print(
    "ROC-AUC:",
    round(roc_auc_score(y_test, y_prob), 3)
)

Accuracy: 52.0 %

Confusion Matrix:
[[78 76]
 [20 26]]

Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.51      0.62       154
           1       0.25      0.57      0.35        46

    accuracy                           0.52       200
   macro avg       0.53      0.54      0.49       200
weighted avg       0.67      0.52      0.56       200

ROC-AUC: 0.525
